In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# Config
# -------------------------
DATASET  = "LINCS"
CSV_PATH = f"{DATASET}_summary.csv"      # 네 파일명/경로에 맞게
OUT_DIR  = "./plots_from_summary/LINCS_noGE"


GENE_MODES = ["random", "ranked", "wilcoxon_union"]
MODALITIES_LINE = ["CP", "GE", "Early Fusion", "Late Fusion"]  # 라인 4개
MODALITIES_BAR  = ["Early Fusion", "Late Fusion"]        # bar x축
METRICS = ["acc", "f1", "auroc", "topk"]

# top_n은 x축 간격 동일하게 하려면 categorical로 찍는게 제일 깔끔
TOPN_ORDER = ["5","10","15","20","25","30","50","75","100","150","200","300","400","978"]

os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# Load
# -------------------------
df = pd.read_csv(CSV_PATH)

# 기본 정리
df = df.copy()
df["dataset"] = df["dataset"].astype(str)
df["top_n"] = df["top_n"].astype(str)
df["gene_mode"] = df["gene_mode"].astype(str).str.lower()
df["modality"] = df["modality"].astype(str)

# dataset filter
df = df[df["dataset"] == DATASET].copy()

# wilcoxon_union이 top_n 일부에서 없을 수도 있으니(또는 너 말대로 10부터 뜨는 등)
# 없는 조합은 그냥 skip하게 둠.

# -------------------------
# Helpers
# -------------------------
def _metric_cols(metric: str):
    return f"{metric}_mean", f"{metric}_std"

def _setup_ax(ax, title, xlabels):
    ax.set_title(title)
    ax.set_xticks(range(len(xlabels)))
    ax.set_xticklabels(xlabels, rotation=0)
    ax.grid(True, axis="y", alpha=0.3)

def plot_lines_4metrics_4lines(df_in: pd.DataFrame, gene_mode: str, out_path: str):
    """
    1 figure:
      - 2x2 subplots: acc, f1, auroc, topk
      - each subplot: 4 lines (CP/GE/EF/LF) over top_n
    """
    sub = df_in[df_in["gene_mode"] == gene_mode].copy()
    if sub.empty:
        print(f"[SKIP] no rows for gene_mode={gene_mode}")
        return

    # top_n order filter (only those present, keep order)
    present_topn = [t for t in TOPN_ORDER if t in set(sub["top_n"])]
    if len(present_topn) == 0:
        print(f"[SKIP] no matching top_n for gene_mode={gene_mode}")
        return

    # Prepare figure
    fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
    axes = axes.ravel()

    for ax, metric in zip(axes, METRICS):
        mcol, scol = _metric_cols(metric)

        # for each modality, build y over present_topn
        for mod in MODALITIES_LINE:
            s2 = sub[sub["modality"] == mod].copy()
            if s2.empty:
                continue

            # align by top_n
            y = []
            for t in present_topn:
                row = s2[s2["top_n"] == t]
                if row.empty:
                    y.append(np.nan)
                else:
                    y.append(float(row.iloc[0][mcol]))
            ax.plot(range(len(present_topn)), y, marker="o", label=mod, markersize=3)

        _setup_ax(ax, metric, present_topn)
        ax.set_ylabel(mcol)
        
 
    Y_LIMS = {
        "acc":   (0.0, 0.35),
        "f1":    (0.0, 0.35),
        "auroc": (0.5, 0.75),
        "topk":  (0.3, 0.45),
    }

    # metric 이름을 어떻게 돌고있든, 각 ax에 적용
    ax.set_ylim(*Y_LIMS[metric])

    # Legend: figure 내부(상단 중앙)에 배치
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(labels), frameon=False)

    fig.suptitle(f"{DATASET} | gene_mode={gene_mode}", y=0.98)
    fig.tight_layout(rect=[0, 0.06, 1, 0.95])  # 위쪽 공간 조금 확보 (legend + suptitle)
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print(f"[SAVE] {out_path}")

def plot_bars_topn_compare(df_in: pd.DataFrame, top_n: str, out_path: str):
    """
    1 figure:
      - 2x2 subplots: acc, f1, auroc, topk
      - x: GE/EF/LF
      - bars: random/ranked/wilcoxon_union
      - CP baseline: dashed horizontal line
      - legend: 밖으로 빼서 겹침 방지
    """
    sub = df_in[df_in["top_n"] == str(top_n)].copy()
    if sub.empty:
        print(f"[SKIP] no rows for top_n={top_n}")
        return

    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    axes = axes.ravel()

    xlabels = MODALITIES_BAR
    x = np.arange(len(xlabels))
    width = 0.22

    # CP baseline(각 metric마다 1개 값): gene_mode에 상관없이 CP가 동일하게 찍힐 수도 있는데,
    # 혹시 여러 행이 있으면 평균으로 처리.
    for ax, metric in zip(axes, METRICS):
        mcol, scol = _metric_cols(metric)

        # CP baseline
        cp_rows = sub[sub["modality"] == "CP"]
        cp_val = np.nan
        if not cp_rows.empty:
            cp_val = float(cp_rows[mcol].mean())
            ax.axhline(cp_val, linestyle="--", linewidth=1.2, label="CP baseline")

        # bars: each gene_mode
        for i, gm in enumerate(GENE_MODES):
            gm_rows = sub[sub["gene_mode"] == gm]
            ys = []
            for mod in xlabels:
                row = gm_rows[gm_rows["modality"] == mod]
                if row.empty:
                    ys.append(np.nan)
                else:
                    ys.append(float(row.iloc[0][mcol]))
            ax.bar(x + (i-1)*width, ys, width=width, label=gm)
        Y_LIMS = {
            "acc":   (0.23, 0.30),
            "f1":    (0.20, 0.28),
            "auroc": (0.60, 0.72),
            "topk":  (0.30, 0.43),
        }
        ax.set_ylim(*Y_LIMS[metric])
        ax.set_title(metric)
        ax.set_xticks(x)
        ax.set_xticklabels(["EF", "LF"])
        ax.set_ylabel(mcol)
        ax.grid(True, axis="y", alpha=0.3)



    # Legend: figure-level로 밖에 배치 (겹침 해결)
    handles, labels = axes[0].get_legend_handles_labels()
    # 중복 제거(옵션)
    uniq = {}
    for h, l in zip(handles, labels):
        uniq[l] = h
    fig.legend(
        uniq.values(), uniq.keys(),
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=4,
        frameon=False,
    )

    fig.suptitle(f"{DATASET} | top_n={top_n} | method comparison (bars)", y=0.98)
    fig.tight_layout(rect=[0, 0.06, 1, 0.95])
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print(f"[SAVE] {out_path}")

# -------------------------
# Run: line plots (ONLY 3 images)
# -------------------------
for gm in GENE_MODES:
    out = os.path.join(OUT_DIR, f"line_{gm}_4metrics_4lines.png")
    plot_lines_4metrics_4lines(df, gm, out)


[SAVE] ./plots_from_summary/LINCS_noGE/line_random_4metrics_4lines.png
[SAVE] ./plots_from_summary/LINCS_noGE/line_ranked_4metrics_4lines.png
[SAVE] ./plots_from_summary/LINCS_noGE/line_wilcoxon_union_4metrics_4lines.png


In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# Config
# -------------------------
DATASET  = "LINCS"
CSV_PATH = f"{DATASET}_summary.csv"      # adjust if needed
OUT_DIR  = "./plots_from_summary/noGE"
os.makedirs(OUT_DIR, exist_ok=True)

GENE_MODES = ["random", "ranked", "wilcoxon_union"]
METRICS = ["f1", "acc"]  # rows
MODALITIES_LINE = ["CP", "Early Fusion", "Late Fusion"]  # lines

TOPN_ORDER = ["5","10","15","20","25","30","50","75","100","150","200","300","400","978"]

OUT_PATH = os.path.join(OUT_DIR, f"{DATASET}_f1_acc.png")

# -------------------------
# Load
# -------------------------
df = pd.read_csv(CSV_PATH).copy()
df["dataset"] = df["dataset"].astype(str)
df["top_n"] = df["top_n"].astype(str)
df["gene_mode"] = df["gene_mode"].astype(str).str.lower()
df["modality"] = df["modality"].astype(str)

df = df[df["dataset"] == DATASET].copy()

def metric_cols(metric: str):
    return f"{metric}_mean", f"{metric}_std"

# -------------------------
# Plot: 2x3 (rows=metrics, cols=gene_modes)
# -------------------------
present_topn = [t for t in TOPN_ORDER if t in set(df["top_n"])]
if len(present_topn) == 0:
    raise ValueError("No matching top_n values found in the dataframe.")

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)

Y_LIMS = {
    "f1":    (0.20, 0.30),   # adjust if you want wider
    "acc": (0.20, 0.30),
}

for c, gm in enumerate(GENE_MODES):
    sub_gm = df[df["gene_mode"] == gm].copy()

    for r, metric in enumerate(METRICS):
        ax = axes[r, c]
        mcol, _ = metric_cols(metric)

        for mod in MODALITIES_LINE:
            s2 = sub_gm[sub_gm["modality"] == mod]
            if s2.empty:
                continue

            y = []
            for t in present_topn:
                row = s2[s2["top_n"] == t]
                y.append(float(row.iloc[0][mcol]) if not row.empty else np.nan)

            ax.plot(
                range(len(present_topn)),
                y,
                marker="o",
                markersize=3,
                label=mod,
            )

        if r == 0:
            ax.set_title(gm)
        if c == 0:
            ax.set_ylabel(f"{metric}_mean")

        ax.set_ylim(*Y_LIMS[metric])
        ax.grid(True, axis="y", alpha=0.3)

# x ticks on BOTH rows (top row labels too)
for r in [0, 1]:
    for ax in axes[r, :]:
        ax.set_xticks(range(len(present_topn)))
        ax.set_xticklabels(present_topn, rotation=0)
        ax.tick_params(axis="x", labelbottom=True)  # force show even when sharex=True

# figure-level legend (avoid overlap)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(labels), frameon=False)

fig.suptitle(f"{DATASET} | f1 & auroc (rows) × gene_mode (cols)", y=0.98)
fig.tight_layout(rect=[0, 0.06, 1, 0.95])
fig.savefig(OUT_PATH, dpi=200)
plt.close(fig)

print(f"[SAVE] {OUT_PATH}")

[SAVE] ./plots_from_summary/noGE/LINCS_f1_acc.png


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def read_stats(path):
    df = pd.read_csv(path)
    acc = pd.to_numeric(df["test/acc"], errors="coerce").dropna().to_numpy()
    f1  = pd.to_numeric(df["test/f1"],  errors="coerce").dropna().to_numpy()
    return {
        "test/acc_mean": acc.mean(), "test/acc_std": acc.std(ddof=1),
        "test/f1_mean":  f1.mean(),  "test/f1_std":  f1.std(ddof=1),
    }

s_m = read_stats("CCA_matched.csv")
s_r = read_stats("CCA_random.csv")

metrics = ["test/acc", "test/f1"]
x = np.arange(len(metrics))
w = 0.28

matched_means = [s_m[f"{m}_mean"] for m in metrics]
matched_stds  = [s_m[f"{m}_std"]  for m in metrics]
random_means  = [s_r[f"{m}_mean"] for m in metrics]
random_stds   = [s_r[f"{m}_std"]  for m in metrics]

fig, ax = plt.subplots(figsize=(6.5, 4.5))
err_kw = dict(capsize=3, elinewidth=1, capthick=1)

ax.bar(x - w/2, matched_means, width=w, yerr=matched_stds, label="matched", error_kw=err_kw)
ax.bar(x + w/2, random_means,  width=w, yerr=random_stds,  label="random",  error_kw=err_kw)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel("mean ± std")
ax.set_title("CCA: matched vs random")
ax.legend()

# auto y-lims with padding
all_y = np.array(matched_means + random_means, float)
all_e = np.array(matched_stds + random_stds, float)
ymin = np.min(all_y - all_e)
ymax = np.max(all_y + all_e)
pad = 0.08 * (ymax - ymin if ymax > ymin else 1.0)
ax.set_ylim(ymin - pad, ymax + pad)

plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'CCA_matched.csv'

In [5]:
# -------------------------
# Run: bar plot for a chosen top_n
# -------------------------
TOPN_PICK = "978"  # 원하는 값으로 바꿔
out_bar = os.path.join(OUT_DIR, f"bar_topn{TOPN_PICK}_4metrics.png")
plot_bars_topn_compare(df, TOPN_PICK, out_bar)

[SAVE] ./plots_from_summary/LINCS_noGE/bar_topn978_4metrics.png
